This supporting notebook preserves the former worked version of 8.0: order sums, moving observation planes, phase-information comparison, sign conventions and the surface-evaluation outlook. It is available for reuse in later lessons; 8.1-8.5 remain planned. The concise chapter introduction is [8.0](../8_0_Introduction_to_Field_Propagation.ipynb).


# Field propagation: retained worked material

**Given the complex optical field at a reference plane, how do we determine the field at another location?**

$$\boxed{\text{Known reference-plane field}\ \longrightarrow\ \text{propagation through a specified medium}\ \longrightarrow\ \text{field at an observation location}}$$

[0.4](../0_4_Propagation_of_Scalar_Optical_Fields.ipynb) provided an early propagation preview. Chapter 8 now develops propagation systematically using the outgoing waves and coordinate representations of [5.1–5.3](../5_0_Grating_Diffraction_in_Three_Dimensions.ipynb), the optical-system distinction in [5.4](../5_4_From_Diffraction_Coordinates_to_Pupils.ipynb), and the interference, finite-beam and complex-coefficient ideas of [7.2](../7_2_Interference_of_Diffraction_Orders_from_a_Grating.ipynb), [7.3](../7_3_Finite_Diffraction_Beams_and_Experimental_Recombination.ipynb) and [7.4](../7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb).

Run independently, in order, with NumPy, Matplotlib and ipywidgets. One experiment moves an observation plane through a prescribed coherent field; a short comparison then shows why its starting intensity is insufficient. No FFT, material-scattering solve or imaging system is needed here.


## 1. State the input before choosing a propagation method

Choose a reference plane $z=z_{\rm ref}$ entirely in the homogeneous outgoing medium, beyond the region where material scattering must be solved. The known data are the **complex field**
$$U_{\rm ref}(x,y)=U(x,y,z_{\rm ref}),$$
or, for a specified discrete set of components,
$$U_{\rm ref}(x,y)=\sum_j C_j(z_{\mathrm{ref}})e^{i(k_{x,j}x+k_{y,j}y)}.$$
Here $j$ can mean a one-dimensional order $m$ or a two-dimensional pair $(m,n)$. Coefficients include magnitude and phase at the stated plane and lateral origin.

| Question | Required model |
| --- | --- |
| What field does the grating produce? | Prescribed transmission or a scattering calculation. |
| How does that known field evolve through a homogeneous region? | Propagation. |
| How does a collecting and imaging system transform it? | An optical-system model with stated assumptions. |

We start with known coefficients. They are not inferred from grating pitch alone. Also impose **no incoming wave from the distant outgoing half-space and no exponentially growing solution there**. A plane field by itself does not distinguish arbitrary forward and backward contributions. The outgoing/decaying condition selects the longitudinal branch. At grazing, the bounded constant solution is retained.

For this experiment, $z_{\rm ref}=0\,\mu\mathrm m$ is a chosen homogeneous-medium reference plane, with lateral origin $(0,0)$. It need not coincide with a material interface. All $C_j(z_{\mathrm{ref}})$ are in units of one reference field amplitude; relative intensity is $I=|U|^2$. These are field units, not flux-normalized efficiencies.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](../0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## 2. Positive distance toward negative z

Retain the later grating convention
$$Ue^{-i\omega t},\qquad z=z_{\rm ref}-s,\qquad s\ge0.$$
Write vacuum and in-medium wavelength explicitly:
$$k_0=\frac{2\pi}{\lambda_0},\qquad k=nk_0=\frac{2\pi}{\lambda},\qquad \lambda=\frac{\lambda_0}{n}.$$
For a homogeneous, isotropic, lossless outgoing medium with real $n>0$,
$$k_{z,j}=-\sqrt{k^2-k_{x,j}^2-k_{y,j}^2}.$$
The square root is the principal complex root, followed by the minus sign:

| Component | Chosen $k_z$ | Factor over positive distance $s$ |
| --- | --- | --- |
| Propagating | Negative real $-\sqrt{k^2-k_x^2-k_y^2}$ | $e^{-ik_zs}$: phase changes, magnitude is 1. |
| Grazing | $0$ | $1$: no normal variation. |
| Evanescent | $-i\kappa$, $\kappa>0$ | $e^{-\kappa s}$: amplitude decays. |

Thus
$$\boxed{U(x,y,z_{\rm ref}-s)=\sum_j C_j(z_{\mathrm{ref}})
 e^{i(k_{x,j}x+k_{y,j}y)}e^{-ik_{z,j}s}}.$$
At $s=0$ this reconstructs the specified reference field. At another plane, its harmonic coefficients are $C_j(s)=C_j(z_{\mathrm{ref}})e^{-ik_{z,j}s}$. A propagating coefficient preserves its magnitude, but not generally its phase.

**Convention bridge to 0.4.** That lesson chooses outgoing $+z$, with $k_z^{(+)}=+\sqrt{\cdots}$ and factor $e^{ik_z^{(+)}s}$. Here $k_z^{(-)}=-k_z^{(+)}$ and the global displacement is $-s$, so $e^{-ik_z^{(-)}s}=e^{ik_z^{(+)}s}$. The multiplier is identical when expressed using positive distance into the respective outgoing side. The existing `scalar_propagation.transfer_function` expects **positive distance into +z**, not signed global displacement. Do not pass `-s` to it. Our order-sum evaluator expects positive $s$ toward $-z$ and uses signed negative $k_z$.


In [ ]:
from pathlib import Path
import sys, importlib
_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p/'wave_optics'/'__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within Grating_Physics.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
import wave_optics
wave_optics.__path__[:] = [str(_support_dir/'wave_optics')]
# Release the previous explorer before reloading its helpers.
if 'propagation_explorer' in globals():
    propagation_explorer.close()
from wave_optics import field_propagation as propagation
from wave_optics import field_propagation_view as propagation_view
importlib.reload(propagation)
importlib.reload(propagation_view)
import numpy as np
from IPython.display import display, Image


## 3. Experiment: move the observation plane

Use normal-incidence order geometry with vacuum wavelength $\lambda_0=2\,\mu\mathrm m$, outgoing index $n=1$, hence in-medium wavelength $\lambda=2\,\mu\mathrm m$, and grating pitch $d=4\,\mu\mathrm m$. The field is uniform along $y$. With $G=2\pi/d$,
$$\mathbf k_0=(0,0,-\pi),\qquad
\mathbf k_{+1}=(\pi/2,0,-\sqrt3\pi/2)\quad\text{rad}/\mu\mathrm m.$$
In this paragraph $\mathbf k_0$ means the **zero-order vector**; the scalar $k_0=2\pi/\lambda_0$ remains the vacuum wavenumber.

The default prescribed input is
$$C_0(z_{\mathrm{ref}})=C_{+1}(z_{\mathrm{ref}})=1,\qquad U_{\rm ref}=1+e^{iGx}.$$
Only these two components are present: their sum **defines** the illustrated input field and is not an exact reconstruction of a binary mask. The single-order option instead uses only $C_{+1}(z_{\mathrm{ref}})=1$.

**Predict, then explore:**

1. Set $s=0$. Reference and observation curves coincide.
2. Increase $s$ to 2, then 4 micrometres. Do individual magnitudes change, or does their relative phase change?
3. Choose **Single order +1**. Move $s$ again: its phase changes, while its magnitude and intensity remain uniform.
4. Reset to compare the two-order field.

The top panels show complex-field magnitude and wrapped phase at the two planes. Phase is omitted where amplitude is below $10^{-8}$ in the stated field units; a zero field has no phase. The lower-left intensity map shows the intervening evolution, with positive $s$ running downward toward $-z$ and a dashed line at the selected plane. The lower-right panel compares relative intensities. All intensity views use the fixed scale 0–4. Blue denotes the reference plane and orange the selected observation plane; the map colors encode intensity, not signed instantaneous field.


In [ ]:
propagation_explorer = propagation_view.FieldPropagationExplorer()
display(propagation_explorer.ui)


### Why do the fringes move?

The two propagating orders retain their transverse wavevectors and unit coefficient magnitudes. Their phase gains are $-k_{z,0}s$ and $-k_{z,+1}s$, which differ. For this pair,
$$I(x,s)=2+2\cos\!\left[Gx-(k_{z,+1}-k_{z,0})s\right].$$
Here $k_{z,+1}-k_{z,0}=\pi(1-\sqrt3/2)>0$. At $s=2\,\mu\mathrm m$, the interference phase offset is approximately $-0.842$ rad, and the maximum initially at $x=0$ shifts to about $x=+0.536\,\mu\mathrm m$. By contrast, the single order acquires only a common phase and keeps $I=1$ everywhere.

A common phase multiplying the whole field changes no intensity; different longitudinal phase gains change the relative phase and therefore the reconstructed local intensity. This is interference during propagation, not a new scattering event or an amplitude attenuation of the individual propagating waves. The intensity averaged over a full transverse period remains constant for the two propagating orders.

The plotted edges are viewing limits. The components extend across the full transverse plane; there are no physical beam edges, grating edges or finite-aperture diffraction in this order-sum model.

### A short evanescent comparison

The same order rule gives $m=3$ with $k_x=3\pi/2>k=\pi$, so $k_z=-i\pi\sqrt5/2$. Its factor is $e^{-\pi\sqrt5s/2}$, rather than a unit-magnitude phase factor. The following calculation prescribes this component separately; it does not add it to the main experiment.


In [ ]:
evanescent = propagation.prescribed_order_data((3,))
for depth in [0., .5, 1.]:  # positive distance toward -z, micrometres
    u = propagation.component_field(evanescent['vectors'], evanescent['coefficients'],
                                    x=0., s=depth)
    print(f's={depth:.1f} um: relative amplitude = {abs(u):.4f}')


## 4. The same starting intensity can hide different fields

Compare two specified coherent reference fields:
$$U_A(x,0)=1+e^{iGx},\qquad U_B(x,0)=1+e^{-iGx}.$$
Their starting intensities are exactly equal:
$$|U_A(x,0)|^2=|U_B(x,0)|^2=2+2\cos(Gx).$$
Their phases differ. Propagate A with orders $(0,+1)$ and B with $(0,-1)$, retaining each order's own $k_x$ and the same outgoing branch. The first orders have equal $k_z$, but the zero order has a different $k_z$. At distance $s$, set $\delta=-(k_{z,1}-k_{z,0})s$; then
$$I_A=2+2\cos(Gx+\delta),\qquad I_B=2+2\cos(-Gx+\delta).$$
At $s=2\,\mu\mathrm m$, $\delta\approx-0.842$ rad. The maxima move in opposite transverse directions, so the two intensity patterns differ at the same observation coordinates. At special distances they may coincide again; the claim is not that they differ at every distance.

**Predict:** can a measurement of the common reference intensity tell us which downstream pattern will appear? Compare the reference phases before looking at the downstream curves.


In [ ]:
display(Image(data=propagation_view.png(
    propagation_view.phase_information_figure(distance=2.))))


<details><summary>Interpret the comparison</summary>

The upper-left curves coincide, but the upper-right phases do not. The lower panels show different downstream fields and intensity patterns. The reference intensity discarded the information needed to choose between them. Propagation therefore needs the complex reference field, or equivalent complex component coefficients, as Chapter 7's coefficient phases anticipated.

</details>

## 5. From discrete orders to a general field

The organizing idea is the same in every representation:

> Decompose the known field, propagate its components, and reconstruct the field.

An ideal periodic field can have a discrete order description. A general transverse field may require a continuous angular spectrum; finite illumination can broaden the angular content around the ideal order locations. The finite Gaussian beams in 7.3 already separated ideal order directions from actual spatial envelopes.

The **angular-spectrum method** is the forthcoming general treatment in 8.2; its Fourier-integral derivation and sampled implementation are deferred there. Later lessons ask when Fresnel and Fraunhofer approximations are appropriate, how periodic orders can produce recurring self-images, and how sampling and convergence constrain numerical predictions. Those questions extend this experiment rather than changing its input/output distinction.

## 6. Outlook: an observation surface need not be parallel

In a homogeneous region, known plane-wave components may also be evaluated on a tilted plane or a specified curved surface. With $\mathbf r_{\rm ref}=(0,0,z_{\rm ref})$,
$$U(\mathbf r)=\sum_j C_j(z_{\mathrm{ref}})e^{i\mathbf k_j\cdot(\mathbf r-\mathbf r_{\rm ref})}.$$
Use the **actual observation coordinates**, with the same reference convention and within the applicable outgoing homogeneous region. A spherical cap is one possible surface.

This field evaluation is different from plotting direction labels on a scaled direction-cosine sphere. A sphere marker describes a direction; evaluating $U$ on a cap includes the spatial phases of all components at each point. A reference cap is not automatically a wavefront of the summed field or of its individual plane waves. Detailed cap geometry, lens transformations and reference-sphere field mappings belong to later work.

## Checkpoint

1. What input does propagation require beyond an intensity image?
2. Why is the outgoing condition needed even when the reference field is known?
3. Can local intensity change while every propagating coefficient magnitude stays constant?
4. Does propagating prescribed coefficients solve the grating's material scattering or the collecting lens's transformation?

<details><summary>Answers</summary>

1. The complex field, including phase, or its equivalent complex component coefficients at a stated plane and origin, plus the medium and propagation condition.
2. The plane field alone does not select arbitrary forward versus backward or growing contributions. The outgoing/decaying condition selects the physical half-space solution.
3. Yes. Different longitudinal wavevectors accumulate different phases, changing interference.
4. No. Scattering supplies the input; homogeneous propagation evolves it; an optical-system model supplies any collection and imaging transformation.

</details>

## Planned Chapter 8 progression

| Notebook | Main question | Status |
| --- | --- | --- |
| **8.0 — Introduction to Field Propagation** | What is known, and what does propagation determine? | Chapter introduction available separately. |
| **8.1 — Propagation of Individual Waves and Diffraction Orders** | How do individual components acquire phase or decay? | Planned. |
| **8.2 — Angular-Spectrum Propagation** | How do we propagate a general complex field? | Planned. |
| **8.3 — Fresnel and Fraunhofer Propagation** | How do useful approximations follow from the general description? | Planned. |
| **8.4 — Periodic Fields and the Talbot Effect** | How can evolving orders produce recurring field patterns? | Planned. |
| **8.5 — Numerical Sampling and Convergence** | How do we establish that a computed result is reliable? | Planned. |

This is unnumbered supporting material retained for later Chapter 8 lessons; it does not implement 8.1 or any other planned lesson. Run `propagation_explorer.close()` to release its controls and image buffer; rerunning the setup cell closes the old explorer before replacing it.

[Previous: 7.4 — Complex order coefficients](../7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb) · [Early preview: 0.4](../0_4_Propagation_of_Scalar_Optical_Fields.ipynb) · [Series overview](../0_0_Foundations_of_Scalar_Optics.ipynb) · [Synthesis toward imaging](../Closing_0_From_Diffraction_to_Imaging.ipynb)
